# Interactive Bubble Mask Editor

A matplotlib-based interactive tool for reviewing, adding, and removing bubble segmentation masks.
This editor works with masks generated by `automatic_mask_generator.ipynb`.

> **GPU recommended:** On Google Colab, go to **Runtime > Change runtime type > T4 GPU** before running. SAM2 runs ~100x faster on GPU and may time out on CPU-only runtimes.

> **Note:** The interactive editor requires a local Jupyter environment with `ipympl` installed (`pip install ipympl`). It does **not** work on Google Colab due to the requirement for `%matplotlib widget`. A non-interactive visualization mode is provided below for Colab users.

## Features
- **Remove mode**: Click on any mask to remove it
- **Add mode**: Left-click for positive points (green), right-click for negative points (red)
- **Bounding box**: Draw a box to prompt SAM2 for a new mask
- **Multiple views**: Raw image, colored masks, blue union overlay
- **Sequence navigation**: Move through image series with Next/Back
- **Undo**: Revert the last add/remove action
- **Zoom**: Mouse scroll to zoom in/out
- **Auto-save**: Exports updated masks (pickle) and segmented overlay (PNG)

> **Reference:** Khojasteh, A.R., van de Water, W. & Westerweel, J. (2024). *Practical object and flow structure segmentation using artificial intelligence.* Experiments in Fluids, 65, 119.

## 1. Setup and Installation
Run the next two cells to install dependencies and download model checkpoints.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("\u26a0 Google Colab detected.")
    print("  The interactive editor requires %matplotlib widget, which is not")
    print("  supported on Colab. You can still view masks using the non-interactive")
    print("  visualization cells at the bottom of this notebook.")
    print()
    print("  For full interactive editing, run this notebook locally with:")
    print("    pip install ipympl")
    print("    jupyter notebook")

try:
    import sam2
    print("SAM2 already installed.")
except ImportError:
    print("Installing SAM2 (this takes ~1 minute)...")
    %pip install -q git+https://github.com/facebookresearch/sam2.git
    print("SAM2 installed.")

if not IN_COLAB:
    try:
        import ipympl
    except ImportError:
        print("Installing ipympl for interactive widgets...")
        %pip install -q ipympl

In [ ]:
import os, subprocess

# --- Resolve repository root (works from Notebooks/ or repo root) ---
if IN_COLAB:
    REPO_ROOT = "."
elif os.path.basename(os.getcwd()) == "Notebooks":
    REPO_ROOT = os.path.abspath("..")
else:
    REPO_ROOT = os.path.abspath(".")

# Download checkpoints (all sizes, since users can switch via radio buttons)
CHECKPOINT_DIR = os.path.join(REPO_ROOT, "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

checkpoints = {
    "sam2.1_hiera_tiny.pt":      "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt",
    "sam2.1_hiera_small.pt":     "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt",
    "sam2.1_hiera_base_plus.pt": "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_base_plus.pt",
    "sam2.1_hiera_large.pt":     "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt",
}
# Download large by default; others are fetched on demand when the user switches model
MODELS_TO_DOWNLOAD = ["sam2.1_hiera_large.pt"]

import urllib.request
for ckpt_name in MODELS_TO_DOWNLOAD:
    ckpt_path = os.path.join(CHECKPOINT_DIR, ckpt_name)
    if not os.path.exists(ckpt_path):
        print(f"Downloading {ckpt_name}...")
        urllib.request.urlretrieve(checkpoints[ckpt_name], ckpt_path)
        print(f"  Saved to {ckpt_path}")
    else:
        print(f"Checkpoint found: {ckpt_path}")

# Download demo images on Colab
DEMO_DIR = os.path.join(REPO_ROOT, "Demo")
if IN_COLAB and not os.path.exists(DEMO_DIR):
    print("Downloading demo images...")
    subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                   "https://github.com/AliRKhojasteh/Bubble_segmentation.git", "_temp_repo"],
                  capture_output=True)
    subprocess.run(["git", "-C", "_temp_repo", "sparse-checkout", "set", "Demo"], capture_output=True)
    if os.path.exists("_temp_repo/Demo"):
        import shutil
        shutil.copytree("_temp_repo/Demo", DEMO_DIR)
        shutil.rmtree("_temp_repo")
        print(f"Demo images ready in '{DEMO_DIR}/'")

print("Setup complete.")

## 2. Configure Interactive Backend

In [ ]:
import sys
if "google.colab" not in sys.modules:
    %matplotlib widget
else:
    %matplotlib inline
    print("Using inline backend (non-interactive mode).")

## 3. Imports and Device Configuration

In [ ]:
import numpy as np
import pickle
from PIL import Image
import torch
import matplotlib.pyplot as plt
from matplotlib.widgets import Button, RadioButtons
import cv2
import copy

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator
from sam2.sam2_image_predictor import SAM2ImagePredictor

# Select computation device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

if device.type == "cpu":
    print("\n** WARNING: Running on CPU. SAM2 mask generation will be very slow. **")
    if IN_COLAB:
        print("   Go to Runtime > Change runtime type > T4 GPU and re-run all cells.")

if device.type == "cuda":
    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()
    if torch.cuda.get_device_properties(0).major >= 8:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

## 4. Visualization Helpers

In [ ]:
def show_anns(anns, ax, colors, borders=True):
    """Display colored segmentation masks with optional borders."""
    if len(anns) == 0:
        return
    sorted_anns = sorted(anns, key=lambda x: x["area"], reverse=True)
    h, w = sorted_anns[0]["segmentation"].shape
    img = np.ones((h, w, 4))
    img[:, :, 3] = 0
    for ann in sorted_anns:
        m = ann["segmentation"]
        color_id = ann["id"] % len(colors)
        img[m] = np.concatenate([colors[color_id], [0.5]])
        if borders:
            contours, _ = cv2.findContours(
                m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE
            )
            contours = [cv2.approxPolyDP(c, epsilon=0.01, closed=True) for c in contours]
            cv2.drawContours(img, contours, -1, (0, 0, 1, 0.4), thickness=1)
    ax.imshow(img)


def generate_overlaid_image(image, anns, colors):
    """Create an RGB overlay blending colored masks onto the original image."""
    image_uint8 = image if image.dtype == np.uint8 else (image * 255).astype(np.uint8)
    sorted_anns = sorted(anns, key=lambda x: x["area"], reverse=True)
    overlay = np.zeros((image.shape[0], image.shape[1], 4), dtype=np.float32)
    for ann in sorted_anns:
        m = ann["segmentation"]
        color_id = ann["id"] % len(colors)
        overlay[m] = np.concatenate([colors[color_id], [0.5]])
    alpha = overlay[:, :, 3:4]
    blended = (image_uint8 / 255.0) * (1 - alpha) + overlay[:, :, :3] * alpha
    blended_uint8 = (blended * 255).astype(np.uint8)
    for ann in sorted_anns:
        m = ann["segmentation"]
        contours, _ = cv2.findContours(
            m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE
        )
        contours = [cv2.approxPolyDP(c, epsilon=0.01, closed=True) for c in contours]
        cv2.drawContours(blended_uint8, contours, -1, (0, 0, 255), thickness=1)
    return blended_uint8

## 5. Interactive Mask Editor

The `InteractiveMaskEditor` class provides a full GUI for reviewing and editing masks. It includes buttons for all operations and supports keyboard shortcuts.

> **Local Jupyter only.** This class requires `%matplotlib widget`. On Colab, skip to Section 7 below for non-interactive viewing.

In [ ]:
class InteractiveMaskEditor:
    """Interactive panel for reviewing and editing bubble segmentation masks.
    
    Args:
        directory: Path to folder containing BXXXX.png and BXXXX_masks.pickle files.
        start_index: Starting frame index (default: 1).
        checkpoint_dir: Path to SAM2 checkpoint directory (default: "checkpoints").
    """
    
    MODEL_CONFIGS = {
        "tiny":      ("configs/sam2.1/sam2.1_hiera_t.yaml",  "sam2.1_hiera_tiny.pt"),
        "small":     ("configs/sam2.1/sam2.1_hiera_s.yaml",  "sam2.1_hiera_small.pt"),
        "base_plus": ("configs/sam2.1/sam2.1_hiera_b+.yaml", "sam2.1_hiera_base_plus.pt"),
        "large":     ("configs/sam2.1/sam2.1_hiera_l.yaml",  "sam2.1_hiera_large.pt"),
    }

    def __init__(self, directory, start_index=1, checkpoint_dir=None):
        self.directory = directory
        self.index = start_index
        if checkpoint_dir is None:
            # Default: look for checkpoints/ relative to the notebook's parent directory
            notebook_dir = os.path.dirname(os.path.abspath("."))
            checkpoint_dir = os.path.join(notebook_dir, "checkpoints")
        self.checkpoint_dir = checkpoint_dir
        self._load_current_image()
        
        self.model_name = "large"
        self._load_model(self.model_name)
        self.predictor.set_image(self.image)
        
        self.mode = "remove"
        self.points = []
        self.box = None
        self.box_start = None
        self.drawing_box = False
        self.bg_mode = 2
        self.view_raw = False
        
        self._assign_ids()
        self.next_id = max((m["id"] for m in self.masks), default=-1) + 1
        self.colors = [np.random.random(3) for _ in range(10000)]
        self.undo_stack = []
        
        self._setup_ui()
        self._connect_events()
        self._redraw()

    def _load_current_image(self):
        image_path = os.path.join(self.directory, f"B{self.index:04d}.png")
        pickle_path = os.path.join(self.directory, f"B{self.index:04d}_masks.pickle")
        if not os.path.exists(image_path):
            raise FileNotFoundError(f"Image not found: {image_path}")
        self.image = np.array(Image.open(image_path).convert("RGB"))
        if os.path.exists(pickle_path):
            with open(pickle_path, "rb") as f:
                self.masks = pickle.load(f)
        else:
            self.masks = []

    def save_current(self, save_png=True):
        """Save masks to pickle and optionally export a segmented PNG overlay."""
        pickle_path = os.path.join(self.directory, f"B{self.index:04d}_masks.pickle")
        with open(pickle_path, "wb") as f:
            pickle.dump(self.masks, f)
        print(f"Saved masks to {pickle_path}")
        if save_png:
            overlaid = generate_overlaid_image(self.image, self.masks, self.colors)
            png_path = os.path.join(self.directory, f"B{self.index:04d}_segmented.png")
            Image.fromarray(overlaid).save(png_path)
            print(f"Saved overlay to {png_path}")

    def _assign_ids(self):
        for i, mask in enumerate(self.masks):
            if "id" not in mask:
                mask["id"] = i

    def _load_model(self, model_name):
        cfg, ckpt_name = self.MODEL_CONFIGS[model_name]
        ckpt_path = os.path.join(self.checkpoint_dir, ckpt_name)
        # Auto-download checkpoint if not present (e.g., user switched model size)
        if not os.path.exists(ckpt_path):
            url = f"https://dl.fbaipublicfiles.com/segment_anything_2/092824/{ckpt_name}"
            print(f"Downloading {ckpt_name}...")
            import urllib.request
            urllib.request.urlretrieve(url, ckpt_path)
            print(f"  Saved to {ckpt_path}")
        self.sam2_model = build_sam2(cfg, ckpt_path, device=device)
        # Note: parameters differ from automatic_mask_generator.ipynb — the
        # interactive editor uses more thorough detection (higher points_per_side,
        # crop_n_layers=1) at the cost of speed, since users process one image
        # at a time and can manually refine results.
        self.mask_generator = SAM2AutomaticMaskGenerator(
            model=self.sam2_model,
            points_per_side=128, points_per_batch=128,
            pred_iou_thresh=0.7, stability_score_thresh=0.92,
            stability_score_offset=0.7, crop_n_layers=1,
            box_nms_thresh=0.7, crop_n_points_downscale_factor=2,
            min_mask_region_area=25.0, use_m2m=True,
        )
        self.predictor = SAM2ImagePredictor(self.sam2_model)

    def _setup_ui(self):
        self.fig = plt.figure(figsize=(12, 12))
        self.ax = self.fig.add_subplot(1, 1, 1)
        plt.subplots_adjust(bottom=0.3, left=0.1, right=0.9, top=0.85)
        
        btn_specs = [
            (0.05, 0.2, "Add",       self._on_add_mode),
            (0.15, 0.2, "Remove",    self._on_remove_mode),
            (0.25, 0.2, "Box",       self._on_draw_box),
            (0.35, 0.2, "Mask",      self._on_generate),
            (0.45, 0.2, "Clear",     self._on_clear),
            (0.55, 0.2, "Bkg",       self._on_toggle_bg),
            (0.65, 0.2, "Raw",       self._on_raw_toggle),
            (0.75, 0.2, "Erase All", self._on_erase_all),
            (0.05, 0.05, "Back",     self._on_back),
            (0.15, 0.05, "Next",     self._on_next),
            (0.25, 0.05, "Save",     self._on_save),
            (0.35, 0.05, "Quit",     self._on_quit),
            (0.45, 0.05, "Undo",     self._on_undo),
        ]
        self.buttons = []
        for x, y, label, callback in btn_specs:
            ax_btn = plt.axes([x, y, 0.08, 0.075])
            btn = Button(ax_btn, label)
            btn.label.set_fontsize(8)
            btn.on_clicked(callback)
            self.buttons.append(btn)
        
        self.rax = plt.axes([0.4, 0.9, 0.2, 0.1])
        self.radio = RadioButtons(self.rax, ("tiny", "small", "base_plus", "large"))
        self.radio.set_active(3)
        self.radio.on_clicked(self._on_model_change)

    def _connect_events(self):
        self.fig.canvas.mpl_connect("button_press_event", self._on_click)
        self.fig.canvas.mpl_connect("key_press_event", self._on_key)
        self.fig.canvas.mpl_connect("scroll_event", self._on_scroll)

    def _redraw(self):
        xlim, ylim = self.ax.get_xlim(), self.ax.get_ylim()
        self.ax.clear()
        
        if self.view_raw:
            self.ax.imshow(self.image)
            self.ax.set_title("Raw Image")
        elif self.bg_mode == 2:
            binary = np.zeros(self.image.shape[:2], dtype=bool)
            for m in self.masks:
                binary |= m["segmentation"]
            overlay = np.zeros((*self.image.shape[:2], 4), dtype=np.float32)
            if np.any(binary):
                overlay[binary] = [0, 0, 1, 0.3]
            alpha = overlay[:, :, 3:]
            blended = (self.image / 255.0) * (1 - alpha) + overlay[:, :, :3] * alpha
            self.ax.imshow((blended * 255).astype(np.uint8))
            self._draw_prompts()
            self.ax.set_title(
                f"Mode: {self.mode} | Model: {self.model_name} | "
                f"Masks: {len(self.masks)} | File: B{self.index:04d}.png | View: Blue Overlay"
            )
        else:
            self.ax.imshow(self.image if self.bg_mode == 0 else np.zeros_like(self.image))
            show_anns(self.masks, self.ax, self.colors)
            self._draw_prompts()
            view_label = "Raw + Masks" if self.bg_mode == 0 else "Masks Only"
            self.ax.set_title(
                f"Mode: {self.mode} | Model: {self.model_name} | "
                f"Masks: {len(self.masks)} | File: B{self.index:04d}.png | View: {view_label}"
            )
        
        if xlim != (0.0, 1.0):
            self.ax.set_xlim(xlim)
            self.ax.set_ylim(ylim)
        self.fig.canvas.draw()

    def _draw_prompts(self):
        if self.points:
            pos = np.array([p[:2] for p in self.points if p[2] == 1])
            neg = np.array([p[:2] for p in self.points if p[2] == 0])
            if len(pos) > 0:
                self.ax.scatter(pos[:, 0], pos[:, 1], c="g", marker="*", s=100)
            if len(neg) > 0:
                self.ax.scatter(neg[:, 0], neg[:, 1], c="r", marker="*", s=100)
        if self.box is not None:
            x0, y0, x1, y1 = self.box
            self.ax.add_patch(plt.Rectangle(
                (x0, y0), x1 - x0, y1 - y0, edgecolor="green", facecolor="none", lw=2
            ))

    def _on_scroll(self, event):
        if event.inaxes != self.ax:
            return
        scale = 1 / 1.2 if event.button == "up" else 1.2
        xlim, ylim = self.ax.get_xlim(), self.ax.get_ylim()
        xdata, ydata = event.xdata, event.ydata
        w = (xlim[1] - xlim[0]) * scale
        h = (ylim[1] - ylim[0]) * scale
        rx = (xlim[1] - xdata) / (xlim[1] - xlim[0])
        ry = (ylim[1] - ydata) / (ylim[1] - ylim[0])
        self.ax.set_xlim(xdata - w * (1 - rx), xdata + w * rx)
        self.ax.set_ylim(ydata - h * (1 - ry), ydata + h * ry)
        self.fig.canvas.draw_idle()

    def _on_click(self, event):
        if event.inaxes != self.ax:
            return
        x, y = int(event.xdata), int(event.ydata)
        
        if self.mode == "remove":
            removed = []
            for i in sorted(range(len(self.masks)), reverse=True):
                seg = self.masks[i]["segmentation"]
                if 0 <= y < seg.shape[0] and 0 <= x < seg.shape[1] and seg[y, x]:
                    removed.append(self.masks.pop(i))
            if removed:
                self.undo_stack.append(("remove", copy.deepcopy(removed)))
            self._redraw()
        elif self.mode == "add":
            if self.drawing_box:
                if self.box_start is None:
                    self.box_start = (x, y)
                else:
                    xs, ys = self.box_start
                    self.box = [min(xs, x), min(ys, y), max(xs, x), max(ys, y)]
                    self.box_start = None
                    self.drawing_box = False
                    self._redraw()
            else:
                label = 1 if event.button == 1 else 0
                self.points.append((x, y, label))
                self._redraw()

    def _on_key(self, event):
        key_map = {
            "a": self._on_add_mode, "m": self._on_remove_mode,
            "d": self._on_draw_box, "enter": self._on_generate,
            "c": self._on_clear, "t": self._on_toggle_bg,
            "e": self._on_erase_all, "q": self._on_quit,
            "u": self._on_undo, "r": self._on_raw_toggle,
            "n": self._on_next, "b": self._on_back, "s": self._on_save,
        }
        if event.key in key_map:
            key_map[event.key](None)

    def _on_add_mode(self, _):
        self.mode = "add"
        self._clear_prompts()
        self._redraw()

    def _on_remove_mode(self, _):
        self.mode = "remove"
        self._clear_prompts()
        self._redraw()

    def _on_draw_box(self, _):
        if self.mode == "add":
            self.drawing_box = not self.drawing_box
            self.box_start = None
            self._redraw()

    def _on_generate(self, _):
        if self.mode != "add" or (not self.points and self.box is None):
            return
        coords = np.array([p[:2] for p in self.points]) if self.points else None
        labels = np.array([p[2] for p in self.points]) if self.points else None
        box = np.array(self.box) if self.box else None
        
        masks, scores, _ = self.predictor.predict(
            point_coords=coords if coords is not None and len(coords) > 0 else None,
            point_labels=labels if labels is not None and len(labels) > 0 else None,
            box=box, multimask_output=False,
        )
        if masks.shape[0] > 0:
            mask_np = (masks[0] > 0).cpu().numpy() if torch.is_tensor(masks) else (masks[0] > 0)
            self.masks.append({
                "segmentation": mask_np,
                "area": int(np.sum(mask_np)),
                "bbox": self.box or [0, 0, 0, 0],
                "predicted_iou": float(scores[0]) if scores.size > 0 else 0.0,
                "stability_score": 0.95,
                "crop_box": [0, 0, self.image.shape[1], self.image.shape[0]],
                "id": self.next_id,
            })
            self.next_id += 1
            self.undo_stack.append(("add", 1))
        self._clear_prompts()
        self._redraw()

    def _on_clear(self, _):
        self._clear_prompts()
        self._redraw()

    def _on_erase_all(self, _):
        if self.masks:
            self.undo_stack.append(("remove", copy.deepcopy(self.masks)))
            self.masks = []
            self._redraw()

    def _on_undo(self, _):
        if not self.undo_stack:
            return
        action, data = self.undo_stack.pop()
        if action == "add":
            for _ in range(data):
                if self.masks:
                    self.masks.pop()
                    self.next_id -= 1
        elif action == "remove":
            self.masks.extend(copy.deepcopy(data))
        self._redraw()

    def _on_toggle_bg(self, _):
        self.bg_mode = (self.bg_mode + 1) % 3
        self._redraw()

    def _on_raw_toggle(self, _):
        self.view_raw = not self.view_raw
        self._redraw()

    def _on_next(self, _):
        self.save_current(save_png=True)
        self.index += 1
        try:
            self._load_current_image()
            self.predictor.set_image(self.image)
            self._reset_state()
            self._redraw()
        except FileNotFoundError:
            print(f"No more images after index {self.index - 1}.")
            self.index -= 1

    def _on_back(self, _):
        if self.index > 1:
            self.index -= 1
            self._load_current_image()
            self.predictor.set_image(self.image)
            self._reset_state()
            self._redraw()

    def _on_save(self, _):
        self.save_current(save_png=True)

    def _on_quit(self, _):
        self.save_current(save_png=True)
        plt.close(self.fig)
        print("Masks saved. Access via editor.masks if needed.")

    def _on_model_change(self, label):
        self.model_name = label
        self._load_model(self.model_name)
        self.predictor.set_image(self.image)
        self._redraw()

    def _clear_prompts(self):
        self.points = []
        self.box = None
        self.box_start = None
        self.drawing_box = False

    def _reset_state(self):
        self._assign_ids()
        self.next_id = max((m["id"] for m in self.masks), default=-1) + 1
        self._clear_prompts()
        self.mode = "remove"
        self.undo_stack = []
        self.ax.set_xlim(0, self.image.shape[1])
        self.ax.set_ylim(self.image.shape[0], 0)

## 6. Keyboard Shortcuts

| Key | Function | Description |
|-----|----------|-------------|
| `a` | Add Mode | Switch to Add mode (left-click = positive, right-click = negative) |
| `m` | Remove Mode | Switch to Remove mode (click masks to delete) |
| `d` | Draw Box | Toggle bounding box drawing |
| `Enter` | Generate | Generate mask from current points/box |
| `u` | Undo | Undo last action |
| `c` | Clear | Clear current points/box selection |
| `e` | Erase All | Delete all masks on current image |
| `s` | Save | Save progress (pickle and PNG) |
| `n` | Next | Save and load next image |
| `b` | Back | Load previous image |
| `t` | Toggle Bkg | Cycle: Raw+Masks → Masks Only → Blue Overlay |
| `r` | Raw Toggle | Toggle raw image view |
| `q` | Quit | Save and close |

## 7. Launch the Editor (Local Jupyter Only)

Set your image directory and starting frame index, then run the cell.
The directory should contain `BXXXX.png` images and optionally `BXXXX_masks.pickle` files from the automatic mask generator.

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
IMAGE_DIRECTORY = DEMO_DIR        # Directory with BXXXX.png and BXXXX_masks.pickle
START_INDEX = 1                    # Starting frame index
# ============================================================

if "google.colab" not in sys.modules:
    editor = InteractiveMaskEditor(IMAGE_DIRECTORY, START_INDEX, CHECKPOINT_DIR)
else:
    print("Interactive editor is not available on Colab.")
    print("Use the non-interactive viewer below instead.")

## 8. Non-Interactive Viewer (Colab Compatible)

View masks for a specific image without the interactive editor. This works on both Colab and local environments.

In [ ]:
import glob

IMAGE_DIRECTORY = DEMO_DIR
FRAME_INDEX = 1  # Change this to view different frames

image_path = os.path.join(IMAGE_DIRECTORY, f"B{FRAME_INDEX:04d}.png")
pickle_path = os.path.join(IMAGE_DIRECTORY, f"B{FRAME_INDEX:04d}_masks.pickle")

if not os.path.exists(image_path):
    print(f"Image not found: {image_path}")
else:
    image = np.array(Image.open(image_path).convert("RGB"))
    
    masks = []
    if os.path.exists(pickle_path):
        with open(pickle_path, "rb") as f:
            masks = pickle.load(f)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Original
    axes[0].imshow(image)
    axes[0].set_title(f"Original: B{FRAME_INDEX:04d}.png")
    axes[0].axis("off")
    
    # Colored masks
    axes[1].imshow(image)
    if masks:
        colors = [np.random.random(3) for _ in range(10000)]
        for i, m in enumerate(masks):
            if "id" not in m:
                m["id"] = i
        show_anns(masks, axes[1], colors)
    axes[1].set_title(f"Colored Masks ({len(masks)} detected)")
    axes[1].axis("off")
    
    # Blue union overlay
    binary = np.zeros(image.shape[:2], dtype=bool)
    for m in masks:
        binary |= m["segmentation"]
    overlay = np.zeros((*image.shape[:2], 4), dtype=np.float32)
    if np.any(binary):
        overlay[binary] = [0, 0, 1, 0.3]
    alpha_ch = overlay[:, :, 3:]
    blended = (image / 255.0) * (1 - alpha_ch) + overlay[:, :, :3] * alpha_ch
    axes[2].imshow((blended * 255).astype(np.uint8))
    axes[2].set_title("Blue Union Overlay")
    axes[2].axis("off")
    
    plt.tight_layout()
    plt.show()
    
    print(f"Frame B{FRAME_INDEX:04d}: {len(masks)} masks detected")